In [ ]:
!pip install -r requirements.txt

In [ ]:
# 개발 환경
# CPU: Intel(R) Xeon(R) Gold 6226R CPU @ 2.90GHz
# RAM: 383GiB System memory
# GPU: NVIDIA GA100 [A100 PCIe 80GB] X 2
# Storage: Samsung SSD 870 4TB + Samsung SSD 970 2TB
# OS: Linux (Ubuntu 22.04.5)

# 라이브러리 버전
# python: 3.8.5
# torch: 2.1.0+cu121
# transformers: 4.37.2
# pandas: 1.1.4
# numpy:1.19.2
# Jupyter notebook: 6.2.0
# 자세한 라이브러리 버전은 requirements.txt를 참고해주세요.

# 모델 링크 (Hugging Face):
# - Kosmos-2-patch14-224: https://huggingface.co/microsoft/kosmos-2-patch14-224
# - FLAN-T5-large: https://huggingface.co/google/flan-t5-large
# fine-tuning 및 추가 데이터셋은 필요하지 않습니다.
# 따라서 허깅 페이스의 가중치 그대로 사용하셔도 무방합니다.

# 전체 흐름:
# 1. 재현성을 위해 랜덤 시드를 고정합니다.
# 2. 사용 가능한 디바이스(CUDA 또는 CPU)를 결정합니다.
# 3. 시각 질문 답변(VQA)을 위한 Kosmos-2 모델과 프로세서를 로드합니다.
# 4. 언어 기반 선택지 결정를 위한 FLAN-T5 모델과 토크나이저를 로드합니다.
# 5. Kosmos-2를 사용하여 이미지로부터 설명적인 답변을 생성하는 함수를 정의합니다.
# 6. 생성된 답변을 기반으로 FLAN-T5를 사용하여 최적의 선택지를 선택하는 함수를 정의합니다.
# 7. 메인 함수에서: test.csv를 읽고, 각 샘플을 처리하여 답변을 생성하고 선택지를 선택한 후, 결과를 submission.csv로 저장합니다.

import torch
from PIL import Image
import pandas as pd
from tqdm import tqdm
from transformers import AutoProcessor, AutoModelForVision2Seq
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import os
import re
import random
import numpy as np

# ----------------------------------------------
# 랜덤 시드 고정
# ----------------------------------------------
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(0)

# --------------------------------------------------
# 디바이스 설정 (CUDA 사용 가능 시 CUDA, 아니면 CPU)
# --------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

# --------------------------------------------------
# Kosmos-2 (VQA) 모델과 프로세서 로드
# 모델 사이즈: 1.66B params
# --------------------------------------------------
vqa_model_name = "microsoft/kosmos-2-patch14-224"
processor = AutoProcessor.from_pretrained(vqa_model_name, trust_remote_code=True)
vqa_model = AutoModelForVision2Seq.from_pretrained(vqa_model_name, trust_remote_code=True).to(device)
vqa_model.eval()

# --------------------------------------------------
# FLAN-T5 (언어 모델)와 토크나이저 로드
# 모델 사이즈: 783M params
# --------------------------------------------------
t5_model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(t5_model_name)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_model_name).to(device)
t5_model.eval()

# -----------------------------------------------------------------
# Kosmos-2를 사용하여 이미지로부터 질문과 연관된 이미지 설명을 생성
# -----------------------------------------------------------------
def generate_answer_with_kosmos(image_path, question, max_tokens=100):
    def extract_clean_answer(text, question):
        # 질문이 존재할 경우 질문 이후 텍스트를 분할하고, grounding 및 기타 특수 태그를 제거합니다.
        if question in text:
            after_question = text.split(question, 1)[-1]
        else:
            after_question = text
        cleaned = re.sub(r"<[^>]+>", "", after_question)
        return cleaned.strip()

    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as e:
        print(f"[Image Error] {image_path} → {e}")
        return "[Image Error]"

    prompt = f"<grounding>An image showing {question} "

    inputs = processor(images=image, text=prompt, return_tensors="pt")

    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        generated_ids = vqa_model.generate(
            pixel_values=inputs["pixel_values"],
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_tokens,
            image_embeds=inputs.get("image_embeds", None),
            image_embeds_position_mask=inputs.get("image_embeds_position_mask", None),
            use_cache=True,
            do_sample=False,
        )
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        answer = extract_clean_answer(generated_text, question)
    return answer

# -------------------------------------------------
# FLAN-T5를 사용하여 가장 관련성 높은 선택지를 결정
# -------------------------------------------------
def flan_infer(question, generated_answer, choices):
    prompt = f"""Question: {question}
Generated Answer: {generated_answer}
Choices:
(A) {choices[0]}
(B) {choices[1]}
(C) {choices[2]}
(D) {choices[3]}
Pick the most relevant answer letter (A/B/C/D):"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = t5_model.generate(**inputs,
                                    max_new_tokens=5,
                                    do_sample=False)
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    match = re.search(r"[A-D]", decoded.upper())
    if match:
        return match.group(0)
    else:
        return "-"

# ----------------------------------------------
# 메인 함수: test.csv 처리 및 submission.csv 생성
# ----------------------------------------------
def main():
    test_df = pd.read_csv("test.csv")

    results = []

    for i, row in enumerate(tqdm(test_df.itertuples(), total=len(test_df))):
        ID = row.ID
        img_path = row.img_path
        question = row.Question
        choices = [getattr(row, c) for c in ["A", "B", "C", "D"]]

        generated_answer = generate_answer_with_kosmos(img_path, question, max_tokens=200)
        selected_letter = flan_infer(question, generated_answer, choices)

        print(f"[{i+1}/{len(test_df)}] ID: {ID}")
        print(f"Question: {question}")
        print(f"Generated Answer (KOSMOS-2): {generated_answer}")
        print(f"Selected Letter (FLAN-T5): {selected_letter}")
        print(f"Choices: A:{choices[0]} | B:{choices[1]} | C:{choices[2]} | D:{choices[3]}\n")

        results.append({"ID": ID, "answer": selected_letter})

    pd.DataFrame(results).to_csv("submission.csv", index=False)
    print("Saved → submission.csv")

if __name__ == "__main__":
    main()